In [1]:
from typing import List, Optional


In [2]:
class DummyRecursiveCharacterTextSplitter:

  def __init__(
      self,
      chunk_size: int = 1000,
      chunk_overlap: int = 200,
      separators: Optional[List[str]] = None,
  ):
    if chunk_overlap >= chunk_size:
      raise ValueError("chunk_overlap must be smaller than chunk_size.")

    self.chunk_size = chunk_size
    self.chunk_overlap = chunk_overlap
    self.separators = separators or ["\n\n", "\n", " ", ""]

  def split_text(self, text: str) -> List[str]:
    return self._split_text(text, self.separators)

  def _split_text(self, text: str, separators: List[str]) -> List[str]:
    if not text:
      return []

    separator = separators[-1]
    new_separators = []
    for i, sep in enumerate(separators):
      if sep == "":
        separator = ""
        break
      if sep in text:
        separator = sep
        new_separators = separators[i + 1 :]
        break

    splits = text.split(separator) if separator != "" else list(text)

    good_splits = []
    for piece in splits:
      if not piece:
        continue
      if len(piece) <= self.chunk_size:
        good_splits.append(piece)
      else:
        if new_separators:
          sub_splits = self._split_text(piece, new_separators)
          good_splits.extend(sub_splits)
        else:
          good_splits.append(piece)

    return self._merge_splits(good_splits, separator)

  def _merge_splits(self, splits: List[str], separator: str) -> List[str]:
    docs: List[str] = []
    current_doc: List[str] = []
    total_len = 0

    for piece in splits:
      sep_len = len(separator) if current_doc and separator else 0
      piece_len = len(piece) + sep_len

      if total_len + piece_len <= self.chunk_size:
        current_doc.append(piece)
        total_len += piece_len
      else:
        if current_doc:
          doc = separator.join(current_doc)
          if doc.strip():
            docs.append(doc)
          while total_len > self.chunk_overlap and len(current_doc) > 1:
            removed = current_doc.pop(0)
            total_len -= len(removed) + (len(separator) if current_doc else 0)

        current_doc.append(piece)
        total_len = sum(len(p) for p in current_doc) + len(separator) * (
            len(current_doc) - 1
        )

    if current_doc:
      doc = separator.join(current_doc)
      if doc.strip():
        docs.append(doc)

    return docs





In [3]:

# Test the splitter
sample_text = (
    "Introduction to RAG systems.\n\n"
    "Chunking is the process of breaking large documents into smaller pieces. "
    "This helps LLMs retrieve relevant context accurately.\n\n"
    "Recursive character text splitting maintains document semantics by"
    " prioritizing paragraphs, "
    "then sentences, and then words before resorting to arbitrary character"
    " splits."
)

splitter = DummyRecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=25)
chunks = splitter.split_text(sample_text)

for idx, chunk in enumerate(chunks, start=1):
  print(f"--- Chunk {idx} (Length: {len(chunk)}) ---")
  print(chunk)

--- Chunk 1 (Length: 28) ---
Introduction to RAG systems.
--- Chunk 2 (Length: 144) ---
Introduction to RAG systems.

Chunking is the process of breaking large documents into smaller pieces. This helps LLMs retrieve relevant context
--- Chunk 3 (Length: 153) ---
Chunking is the process of breaking large documents into smaller pieces. This helps LLMs retrieve relevant context

retrieve relevant context accurately.
--- Chunk 4 (Length: 155) ---
retrieve relevant context accurately.

Recursive character text splitting maintains document semantics by prioritizing paragraphs, then sentences, and then
--- Chunk 5 (Length: 196) ---
Recursive character text splitting maintains document semantics by prioritizing paragraphs, then sentences, and then

then sentences, and then words before resorting to arbitrary character splits.
